In [4]:
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load environment variables from .env file
load_dotenv()

# Get database credentials from environment variables
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# Create database connection string
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Create database engine
engine = create_engine(DATABASE_URL)

print("Environment variables loaded successfully")
print(f"Connected to: {DB_NAME} on {DB_HOST}:{DB_PORT}")

Environment variables loaded successfully
Connected to: humanitarian_db on localhost:5432


In [5]:
# List of tables to preview
tables = [
    'cleaned_data.health_facilities',
    'cleaned_data.health_facility_type']

dataframes = {}

for table in tables:    
    query = f"SELECT * FROM {table};"
    # Extract just the table name (everything after the dot)
    table_name = table.split('.')[-1]
    dataframes[table_name] = pd.read_sql(query, engine)
    print(f"Loaded {table_name}")
    
print("All tables loaded successfully")

Loaded health_facilities
Loaded health_facility_type
All tables loaded successfully


In [7]:
# CELL 1: Initial Data Overview
print("="*80)
print("INITIAL DATA OVERVIEW")
print("="*80)

# Extract dataframes from dictionary
health_facilities = dataframes['health_facilities']
health_facility_type = dataframes['health_facility_type']

# Dataset A - health_facilities
print("\nDataset A: health_facilities")
print("-" * 40)
print(f"Shape: {health_facilities.shape}")
print(f"Columns: {health_facilities.columns.tolist()}")
print(f"\nData types:")
print(health_facilities.dtypes)
print(f"\nFirst 3 rows:")
print(health_facilities.head(3))
print(f"\nBasic statistics:")
print(health_facilities.describe(include='all'))

# Dataset B - health_facility_type
print("\n" + "="*40)
print("Dataset B: health_facility_type")
print("-" * 40)
print(f"Shape: {health_facility_type.shape}")
print(f"Columns: {health_facility_type.columns.tolist()}")
print(f"\nData types:")
print(health_facility_type.dtypes)
print(f"\nFirst 3 rows:")
print(health_facility_type.head(3))
print(f"\nBasic statistics:")
print(health_facility_type.describe(include='all'))

INITIAL DATA OVERVIEW

Dataset A: health_facilities
----------------------------------------
Shape: (1988, 10)
Columns: ['state', 'state_code', 'county', 'county_code', 'payam', 'payam_code', 'facility_name', 'latitude', 'longitude', 'missing_coordinates']

Data types:
state                   object
state_code              object
county                  object
county_code             object
payam                   object
payam_code              object
facility_name           object
latitude               float64
longitude              float64
missing_coordinates       bool
dtype: object

First 3 rows:
   state state_code     county county_code      payam payam_code  \
0  unity       SS06  abiemnhom      SS0601  abiemnhom   SS060101   
1  unity       SS06  abiemnhom      SS0601  abiemnhom   SS060101   
2  unity       SS06  abiemnhom      SS0601  abiemnhom   SS060101   

   facility_name  latitude  longitude  missing_coordinates  
0  abiemnom phcc  9.398740  28.823400                Fals

In [8]:
# CELL 2: Administrative Hierarchy Validation - State Level
print("="*80)
print("ADMINISTRATIVE HIERARCHY VALIDATION: STATE LEVEL")
print("="*80)

# Dataset A - State to State_Code mapping
print("\nDataset A: State → State_Code Validation")
print("-" * 40)
state_mapping_a = health_facilities.groupby('state_code')['state'].agg(['nunique', 'unique'])
state_issues_a = state_mapping_a[state_mapping_a['nunique'] > 1]

print(f"Total unique State_Code values: {len(state_mapping_a)}")
print(f"Total unique State values: {health_facilities['state'].nunique()}")
print(f"One code → multiple states: {len(state_issues_a)}")

if len(state_issues_a) > 0:
    print("\nISSUE FOUND - Codes mapping to multiple states:")
    for code, row in state_issues_a.iterrows():
        print(f"  Code '{code}' → {row['unique']}")

# Dataset B - State to State_Code mapping
print("\nDataset B: State → State_Code Validation")
print("-" * 40)
state_mapping_b = health_facility_type.groupby('State_Code')['State'].agg(['nunique', 'unique'])
state_issues_b = state_mapping_b[state_mapping_b['nunique'] > 1]

print(f"Total unique State_Code values: {len(state_mapping_b)}")
print(f"Total unique State values: {health_facility_type['State'].nunique()}")
print(f"One code → multiple states: {len(state_issues_b)}")

if len(state_issues_b) > 0:
    print("\nCRITICAL ISSUE - Codes mapping to multiple states:")
    for code, row in state_issues_b.iterrows():
        print(f"  Code '{code}' → {row['unique']}")
else:
    print("All State_Code values map to exactly one State")

# Show the actual mapping for Dataset B
print("\nState → State_Code mapping in Dataset B:")
state_code_mapping = health_facility_type.groupby('State')['State_Code'].agg(['nunique', 'unique'])
state_code_issues = state_code_mapping[state_code_mapping['nunique'] > 1]

if len(state_code_issues) > 0:
    print("ISSUE - Multiple codes for same state:")
    for state, row in state_code_issues.iterrows():
        print(f"  '{state}' → {row['unique']}")
else:
    print("Each state has exactly one State_Code")
    # Display the clean mapping
    clean_mapping = health_facility_type.groupby('State')['State_Code'].first()
    print("\nClean State → State_Code mapping:")
    for state, code in clean_mapping.items():
        print(f"  {state}: {code}")

ADMINISTRATIVE HIERARCHY VALIDATION: STATE LEVEL

Dataset A: State → State_Code Validation
----------------------------------------
Total unique State_Code values: 11
Total unique State values: 11
One code → multiple states: 0

Dataset B: State → State_Code Validation
----------------------------------------
Total unique State_Code values: 19
Total unique State values: 12
One code → multiple states: 10

CRITICAL ISSUE - Codes mapping to multiple states:
  Code 'SS01' → ['Abyei Adminstrative Area' 'Central Equatoria']
  Code 'SS02' → ['Abyei Adminstrative Area' 'Eastern Equatoria']
  Code 'SS03' → ['Jonglei' 'Abyei Adminstrative Area']
  Code 'SS04' → ['Lakes' 'Abyei Adminstrative Area']
  Code 'SS05' → ['Northern Bahr El Ghazal' 'Abyei Adminstrative Area']
  Code 'SS06' → ['UNITY' 'Unity' 'Abyei Adminstrative Area']
  Code 'SS07' → ['Upper Nile' 'Abyei Adminstrative Area']
  Code 'SS08' → ['Warrap' 'Abyei Adminstrative Area']
  Code 'SS09' → ['Western Bahr El Ghazal' 'Abyei Adminstrati

In [9]:
# CELL 3: Administrative Hierarchy Validation - County Level
print("="*80)
print("ADMINISTRATIVE HIERARCHY VALIDATION: COUNTY LEVEL")
print("="*80)

# Dataset A - County to County_Code mapping
print("\nDataset A: County → County_Code Validation")
print("-" * 40)
county_mapping_a = health_facilities.groupby('county_code')['county'].agg(['nunique', 'unique'])
county_issues_a = county_mapping_a[county_mapping_a['nunique'] > 1]

print(f"Total unique County_Code values: {len(county_mapping_a)}")
print(f"Total unique County values: {health_facilities['county'].nunique()}")
print(f"One code → multiple counties: {len(county_issues_a)}")

if len(county_issues_a) > 0:
    print("\nISSUE FOUND - County codes mapping to multiple counties:")
    for code, row in county_issues_a.head(10).iterrows():
        print(f"  Code '{code}' → {row['unique']}")
else:
    print("All County_Code values map to exactly one County")

# Check if each County belongs to exactly one State
print("\nCounty → State relationship validation:")
county_state_a = health_facilities.groupby('county')['state'].agg(['nunique', 'unique'])
county_state_issues_a = county_state_a[county_state_a['nunique'] > 1]

print(f"Counties spanning multiple states: {len(county_state_issues_a)}")
if len(county_state_issues_a) > 0:
    print("ISSUE - Counties in multiple states:")
    for county, row in county_state_issues_a.head(5).iterrows():
        print(f"  '{county}' → {row['unique']}")

# Dataset B - County to County_Code mapping
print("\n" + "="*40)
print("Dataset B: County → County_Code Validation")
print("-" * 40)
county_mapping_b = health_facility_type.groupby('County_Code')['County'].agg(['nunique', 'unique'])
county_issues_b = county_mapping_b[county_mapping_b['nunique'] > 1]

print(f"Total unique County_Code values: {len(county_mapping_b)}")
print(f"Total unique County values: {health_facility_type['County'].nunique()}")
print(f"One code → multiple counties: {len(county_issues_b)}")

if len(county_issues_b) > 0:
    print("\nCRITICAL ISSUE - County codes mapping to multiple counties:")
    for code, row in county_issues_b.head(10).iterrows():
        print(f"  Code '{code}' → {row['unique']}")
else:
    print("All County_Code values map to exactly one County")

# Check Abyei Administrative Area issue at County level
print("\nInvestigating Abyei Administrative Area at County level:")
abyei_counties_a = health_facilities[health_facilities['state'] == 'Abyei Administrative Area']
abyei_counties_b = health_facility_type[health_facility_type['State'] == 'Abyei Adminstrative Area']

print(f"Dataset A - Records in Abyei: {len(abyei_counties_a)}")
print(f"Dataset B - Records in Abyei: {len(abyei_counties_b)}")

if len(abyei_counties_b) > 0:
    print(f"\nDataset B - Counties in Abyei records:")
    print(abyei_counties_b['County'].value_counts().head(10))

ADMINISTRATIVE HIERARCHY VALIDATION: COUNTY LEVEL

Dataset A: County → County_Code Validation
----------------------------------------
Total unique County_Code values: 79
Total unique County values: 79
One code → multiple counties: 0
All County_Code values map to exactly one County

County → State relationship validation:
Counties spanning multiple states: 0

Dataset B: County → County_Code Validation
----------------------------------------
Total unique County_Code values: 79
Total unique County values: 79
One code → multiple counties: 0
All County_Code values map to exactly one County

Investigating Abyei Administrative Area at County level:
Dataset A - Records in Abyei: 0
Dataset B - Records in Abyei: 19

Dataset B - Counties in Abyei records:
County
Abyei AA    19
Name: count, dtype: int64


In [10]:
# CELL 4: Administrative Hierarchy Validation - Payam Level
print("="*80)
print("ADMINISTRATIVE HIERARCHY VALIDATION: PAYAM LEVEL")
print("="*80)

# Dataset A - Payam to Payam_Code mapping
print("\nDataset A: Payam → Payam_Code Validation")
print("-" * 40)
payam_mapping_a = health_facilities.groupby('payam_code')['payam'].agg(['nunique', 'unique'])
payam_issues_a = payam_mapping_a[payam_mapping_a['nunique'] > 1]

print(f"Total unique Payam_Code values: {len(payam_mapping_a)}")
print(f"Total unique Payam values: {health_facilities['payam'].nunique()}")
print(f"One code → multiple payams: {len(payam_issues_a)}")

if len(payam_issues_a) > 0:
    print("\nISSUE FOUND - Payam codes mapping to multiple payams:")
    for code, row in payam_issues_a.head(5).iterrows():
        print(f"  Code '{code}' → {row['unique']}")
else:
    print("All Payam_Code values map to exactly one Payam")

# Check Payam completeness
print(f"\nPayam completeness:")
payam_missing_a = health_facilities['payam'].isna().sum()
payam_code_missing_a = health_facilities['payam_code'].isna().sum()
print(f"Dataset A - Missing Payam: {payam_missing_a} ({payam_missing_a/len(health_facilities)*100:.1f}%)")
print(f"Dataset A - Missing Payam_Code: {payam_code_missing_a} ({payam_code_missing_a/len(health_facilities)*100:.1f}%)")

# Dataset B - Payam to Payam_Code mapping
print("\n" + "="*40)
print("Dataset B: Payam → Payam_Code Validation")
print("-" * 40)

# Note: Dataset B has trailing space in column name
payam_mapping_b = health_facility_type.groupby('Payam_Code ')['Payam'].agg(['nunique', 'unique'])
payam_issues_b = payam_mapping_b[payam_mapping_b['nunique'] > 1]

print(f"Total unique Payam_Code values: {len(payam_mapping_b)}")
print(f"Total unique Payam values: {health_facility_type['Payam'].nunique()}")
print(f"One code → multiple payams: {len(payam_issues_b)}")

if len(payam_issues_b) > 0:
    print("\nISSUE FOUND - Payam codes mapping to multiple payams:")
    for code, row in payam_issues_b.head(5).iterrows():
        print(f"  Code '{code}' → {row['unique']}")
else:
    print("All Payam_Code values map to exactly one Payam")

# Check Payam completeness
payam_missing_b = health_facility_type['Payam'].isna().sum()
payam_code_missing_b = health_facility_type['Payam_Code '].isna().sum()
print(f"\nPayam completeness:")
print(f"Dataset B - Missing Payam: {payam_missing_b} ({payam_missing_b/len(health_facility_type)*100:.1f}%)")
print(f"Dataset B - Missing Payam_Code: {payam_code_missing_b} ({payam_code_missing_b/len(health_facility_type)*100:.1f}%)")

# Cross-check: Does Payam belong to correct County/State?
print("\nPayam → County relationship validation:")
payam_county_a = health_facilities.groupby('payam')['county'].agg(['nunique', 'unique'])
payam_county_issues_a = payam_county_a[payam_county_a['nunique'] > 1]

print(f"Dataset A - Payams spanning multiple counties: {len(payam_county_issues_a)}")
if len(payam_county_issues_a) > 0:
    print("ISSUE - Payams in multiple counties:")
    for payam, row in payam_county_issues_a.head(5).iterrows():
        print(f"  '{payam}' → {row['unique']}")

payam_county_b = health_facility_type.groupby('Payam')['County'].agg(['nunique', 'unique'])
payam_county_issues_b = payam_county_b[payam_county_b['nunique'] > 1]

print(f"\nDataset B - Payams spanning multiple counties: {len(payam_county_issues_b)}")
if len(payam_county_issues_b) > 0:
    print("ISSUE - Payams in multiple counties:")
    for payam, row in payam_county_issues_b.head(5).iterrows():
        print(f"  '{payam}' → {row['unique']}")

ADMINISTRATIVE HIERARCHY VALIDATION: PAYAM LEVEL

Dataset A: Payam → Payam_Code Validation
----------------------------------------
Total unique Payam_Code values: 476
Total unique Payam values: 478
One code → multiple payams: 0
All Payam_Code values map to exactly one Payam

Payam completeness:
Dataset A - Missing Payam: 23 (1.2%)
Dataset A - Missing Payam_Code: 30 (1.5%)

Dataset B: Payam → Payam_Code Validation
----------------------------------------
Total unique Payam_Code values: 471
Total unique Payam values: 472
One code → multiple payams: 1

ISSUE FOUND - Payam codes mapping to multiple payams:
  Code 'SS070103' → ['Akoka' 'Bianythiang']

Payam completeness:
Dataset B - Missing Payam: 0 (0.0%)
Dataset B - Missing Payam_Code: 0 (0.0%)

Payam → County relationship validation:
Dataset A - Payams spanning multiple counties: 3
ISSUE - Payams in multiple counties:
  'aweil town' → ['aweil west' 'aweil centre']
  'mariem east' → ['aweil west' 'aweil centre']
  'tambura' → ['nagero' '

In [11]:
# CELL 5: Cross-Table Comparison - Facility Overlap
print("="*80)
print("CROSS-TABLE COMPARISON: FACILITY OVERLAP")
print("="*80)

# Clean facility names for comparison
def clean_name(name):
    if pd.isna(name):
        return None
    return str(name).strip().upper().replace('  ', ' ')

# Create standardized facility identifiers
health_facilities['facility_name_clean'] = health_facilities['facility_name'].apply(clean_name)
health_facility_type['facility_name_clean'] = health_facility_type['Facility_Name'].apply(clean_name)

# Find common facilities
facilities_a = set(health_facilities['facility_name_clean'].dropna())
facilities_b = set(health_facility_type['facility_name_clean'].dropna())

common_facilities = facilities_a.intersection(facilities_b)
only_in_a = facilities_a - facilities_b
only_in_b = facilities_b - facilities_a

print(f"\nFacility Counts:")
print(f"Dataset A facilities: {len(facilities_a):,}")
print(f"Dataset B facilities: {len(facilities_b):,}")
print(f"Common facilities: {len(common_facilities):,}")
print(f"Facilities only in A: {len(only_in_a):,}")
print(f"Facilities only in B: {len(only_in_b):,}")

print(f"\nOverlap Statistics:")
print(f"Dataset A overlap: {len(common_facilities)/len(facilities_a)*100:.1f}% of A is in B")
print(f"Dataset B overlap: {len(common_facilities)/len(facilities_b)*100:.1f}% of B is in A")

# Sample of unique facilities
print(f"\nSample of facilities only in Dataset A (first 10):")
for i, facility in enumerate(list(only_in_a)[:10], 1):
    print(f"  {i}. {facility}")

print(f"\nSample of facilities only in Dataset B (first 10):")
for i, facility in enumerate(list(only_in_b)[:10], 1):
    print(f"  {i}. {facility}")

# Check for potential name matching issues
print(f"\nPotential Name Matching Issues:")
print(f"Dataset A has {len(only_in_a):,} facilities not in Dataset B")
print(f"Dataset B has {len(only_in_b):,} facilities not in Dataset A")
print(f"\nThis could mean:")
print("  - One dataset has more up-to-date facilities")
print("  - Facility naming conventions differ")
print("  - Some facilities are incorrectly named in one dataset")
print("  - These are genuinely different sets of facilities")

CROSS-TABLE COMPARISON: FACILITY OVERLAP

Facility Counts:
Dataset A facilities: 1,947
Dataset B facilities: 1,480
Common facilities: 1,236
Facilities only in A: 711
Facilities only in B: 244

Overlap Statistics:
Dataset A overlap: 63.5% of A is in B
Dataset B overlap: 83.5% of B is in A

Sample of facilities only in Dataset A (first 10):
  1. RUBEKE PHCU
  2. MASIYA PHCU
  3. LWOKI PHCU
  4. MANGANG PHCU
  5. MAKUR PHCU
  6. NABIA PAI PHCU
  7. MALUALKON PHCC
  8. BOMA PHCC
  9. PAKUR PHCU
  10. AKUAK - RAK PHCU

Sample of facilities only in Dataset B (first 10):
  1. WUJI PHCC
  2. MAPEL MILIATRY PHCC
  3. ANGUORTH PHCU
  4. HIS HOUSE OF HOPE HOSPITAL
  5. POLICE HQS PHCC
  6. YIDA CENTRE A PHCC
  7. PACHIDI PHCU
  8. IMEHEJEK COUNTY HOSPITAL
  9. NAITA PHCU
  10. BIEY PHCU

Potential Name Matching Issues:
Dataset A has 711 facilities not in Dataset B
Dataset B has 244 facilities not in Dataset A

This could mean:
  - One dataset has more up-to-date facilities
  - Facility naming con

In [13]:
# CELL 7: Cross-Table Comparison - Case-Insensitive Administrative Comparison
print("="*80)
print("CROSS-TABLE COMPARISON: CASE-INSENSITIVE ADMINISTRATIVE MATCH")
print("="*80)

# Create case-insensitive versions for comparison
comparison_df['state_A_upper'] = comparison_df['state'].str.upper()
comparison_df['state_B_upper'] = comparison_df['State'].str.upper()
comparison_df['county_A_upper'] = comparison_df['county'].str.upper()
comparison_df['county_B_upper'] = comparison_df['County'].str.upper()
comparison_df['payam_A_upper'] = comparison_df['payam'].str.upper()
comparison_df['payam_B_upper'] = comparison_df['Payam'].str.upper()

# Check case-insensitive matches
state_match_ci = (comparison_df['state_A_upper'] == comparison_df['state_B_upper']).sum()
county_match_ci = (comparison_df['county_A_upper'] == comparison_df['county_B_upper']).sum()
payam_match_ci = (comparison_df['payam_A_upper'] == comparison_df['payam_B_upper']).sum()

print(f"Case-Insensitive Administrative Agreement for Common Facilities:")
print(f"  State match: {state_match_ci:,} ({state_match_ci/len(comparison_df)*100:.1f}%)")
print(f"  County match: {county_match_ci:,} ({county_match_ci/len(comparison_df)*100:.1f}%)")
print(f"  Payam match: {payam_match_ci:,} ({payam_match_ci/len(comparison_df)*100:.1f}%)")

# Find actual disagreements (case-insensitive)
state_disagreements_ci = comparison_df[comparison_df['state_A_upper'] != comparison_df['state_B_upper']]
county_disagreements_ci = comparison_df[comparison_df['county_A_upper'] != comparison_df['county_B_upper']]
payam_disagreements_ci = comparison_df[comparison_df['payam_A_upper'] != comparison_df['payam_B_upper']]

print(f"\nActual Administrative Disagreements (case-insensitive):")
print(f"  State disagreements: {len(state_disagreements_ci):,}")
print(f"  County disagreements: {len(county_disagreements_ci):,}")
print(f"  Payam disagreements: {len(payam_disagreements_ci):,}")

# Sample actual state disagreements
if len(state_disagreements_ci) > 0:
    print(f"\nSample ACTUAL State disagreements (first 10):")
    for idx, row in state_disagreements_ci.head(10).iterrows():
        print(f"  {row['facility_name_clean']}:")
        print(f"    Dataset A: {row['state']}")
        print(f"    Dataset B: {row['State']}")

# Check if casing is the only issue
print(f"\nCasing Pattern Analysis:")
print(f"Dataset A state names - all lowercase: {comparison_df['state'].str.islower().all()}")
print(f"Dataset B state names - mixed case: {not comparison_df['State'].str.islower().all()}")

# Show state name variants
print(f"\nState Name Variants (common facilities):")
state_variants_a = set(comparison_df['state'].unique())
state_variants_b = set(comparison_df['State'].unique())
print(f"Dataset A unique states: {len(state_variants_a)}")
print(f"Dataset B unique states: {len(state_variants_b)}")
print(f"Common states (case-insensitive): {len(set([s.upper() for s in state_variants_a]).intersection(set([s.upper() for s in state_variants_b])))}")

# Show mapping of state names
print(f"\nState Name Mapping (Dataset A → Dataset B):")
state_mapping = {}
for state_a in state_variants_a:
    matching_states = [s for s in state_variants_b if s.upper() == state_a.upper()]
    if matching_states:
        state_mapping[state_a] = matching_states[0]
    else:
        state_mapping[state_a] = "NO MATCH"

for state_a, state_b in list(state_mapping.items())[:10]:
    print(f"  '{state_a}' → '{state_b}'")

CROSS-TABLE COMPARISON: CASE-INSENSITIVE ADMINISTRATIVE MATCH
Case-Insensitive Administrative Agreement for Common Facilities:
  State match: 533 (39.7%)
  County match: 1,199 (89.2%)
  Payam match: 1,140 (84.8%)

Actual Administrative Disagreements (case-insensitive):
  State disagreements: 811
  County disagreements: 145
  Payam disagreements: 204

Sample ACTUAL State disagreements (first 10):
  ABYEI CIVIL HOSPITAL:
    Dataset A: abyei region
    Dataset B: Abyei Adminstrative Area
  ABYEI PHCC:
    Dataset A: abyei region
    Dataset B: Abyei Adminstrative Area
  AGANY TOK PHCU:
    Dataset A: abyei region
    Dataset B: Abyei Adminstrative Area
  AGOK PHCC:
    Dataset A: abyei region
    Dataset B: Western Bahr El Ghazal
  AGOK PHCC:
    Dataset A: abyei region
    Dataset B: Abyei Adminstrative Area
  AMENTH BEK HOSPITAL:
    Dataset A: abyei region
    Dataset B: Abyei Adminstrative Area
  AMIET PHCU:
    Dataset A: abyei region
    Dataset B: Abyei Adminstrative Area
  AWAL P

In [14]:
# CELL 8: State Name Standardization Analysis
print("="*80)
print("STATE NAME STANDARDIZATION ANALYSIS")
print("="*80)

# Create a mapping of state names from both datasets
states_a = sorted(health_facilities['state'].unique())
states_b = sorted(health_facility_type['State'].unique())

print("Dataset A - Unique State Names:")
for state in states_a:
    count = len(health_facilities[health_facilities['state'] == state])
    print(f"  '{state}': {count} facilities")

print("\nDataset B - Unique State Names:")
for state in states_b:
    count = len(health_facility_type[health_facility_type['State'] == state])
    print(f"  '{state}': {count} facilities")

# Attempt to match states
print("\n" + "="*40)
print("Attempting State Name Matching:")

# Known full state names for reference
full_state_names = {
    'cent eq': 'Central Equatoria',
    'east eq': 'Eastern Equatoria',
    'west eq': 'Western Equatoria',
    'north bahr': 'Northern Bahr El Ghazal',
    'west bahr': 'Western Bahr El Ghazal',
    'unity': 'Unity',
    'warrap': 'Warrap',
    'jonglei': 'Jonglei',
    'upper nile': 'Upper Nile',
    'lakes': 'Lakes',
    'abyei region': 'Abyei Administrative Area'
}

print("\nProposed mapping for Dataset A abbreviated names:")
for abbr, full in full_state_names.items():
    if abbr in states_a:
        print(f"  '{abbr}' → '{full}'")
        # Check if full name exists in Dataset B
        if full in states_b:
            print(f"    ✓ '{full}' found in Dataset B")
        else:
            print(f"    ✗ '{full}' NOT found in Dataset B")

# Check what's actually in Dataset B for these states
print("\nActual Dataset B state names that correspond to Dataset A abbreviations:")
for abbr, full in full_state_names.items():
    if abbr in states_a:
        # Find closest match in Dataset B
        matching_b = [s for s in states_b if full.upper() in s.upper() or s.upper() in full.upper()]
        if matching_b:
            print(f"  '{abbr}' → matches: {matching_b}")
        else:
            print(f"  '{abbr}' → NO MATCH found")

# Analyze the "abyei region" issue
print("\n" + "="*40)
print("Abyei Region Analysis:")
abyei_a = health_facilities[health_facilities['state'] == 'abyei region']
abyei_b = health_facility_type[health_facility_type['State'] == 'Abyei Adminstrative Area']

print(f"Dataset A - Records in 'abyei region': {len(abyei_a)}")
print(f"Dataset B - Records in 'Abyei Adminstrative Area': {len(abyei_b)}")

# Check if these are the same facilities
abyei_names_a = set(abyei_a['facility_name_clean'])
abyei_names_b = set(abyei_b['facility_name_clean'])
abyei_common = abyei_names_a.intersection(abyei_names_b)

print(f"\nAbyei facility overlap:")
print(f"  Facilities only in A: {len(abyei_names_a - abyei_names_b)}")
print(f"  Facilities only in B: {len(abyei_names_b - abyei_names_a)}")
print(f"  Facilities in both: {len(abyei_common)}")

if len(abyei_common) > 0:
    print("\nSample Abyei facilities in both datasets:")
    for name in list(abyei_common)[:5]:
        print(f"  {name}")

STATE NAME STANDARDIZATION ANALYSIS
Dataset A - Unique State Names:
  'abyei region': 32 facilities
  'cent eq': 289 facilities
  'east eq': 225 facilities
  'jonglei': 203 facilities
  'lakes': 119 facilities
  'north bahr': 187 facilities
  'unity': 168 facilities
  'upper nile': 221 facilities
  'warrap': 162 facilities
  'west bahr': 118 facilities
  'west eq': 264 facilities

Dataset B - Unique State Names:
  'Abyei Adminstrative Area': 19 facilities
  'Central Equatoria': 161 facilities
  'Eastern Equatoria': 186 facilities
  'Jonglei': 159 facilities
  'Lakes': 118 facilities
  'Northern Bahr El Ghazal': 174 facilities
  'UNITY': 87 facilities
  'Unity': 25 facilities
  'Upper Nile': 134 facilities
  'Warrap': 174 facilities
  'Western Bahr El Ghazal': 90 facilities
  'Western Equatoria': 186 facilities

Attempting State Name Matching:

Proposed mapping for Dataset A abbreviated names:
  'cent eq' → 'Central Equatoria'
    ✓ 'Central Equatoria' found in Dataset B
  'east eq' → '

In [15]:
# CELL 9: Facility Identity Investigation - Facilities_Code Analysis
print("="*80)
print("FACILITY IDENTITY INVESTIGATION: Facilities_Code")
print("="*80)

# Check Facilities_Code uniqueness
print("\nDataset B: Facilities_Code Analysis")
print("-" * 40)
facilities_code_unique = health_facility_type['Facilities_Code'].nunique()
facilities_code_total = len(health_facility_type)

print(f"Total rows: {facilities_code_total:,}")
print(f"Unique codes: {facilities_code_unique:,}")
print(f"Duplicate rate: {(facilities_code_total - facilities_code_unique) / facilities_code_total * 100:.1f}%")
print(f"Is Facilities_Code a true business key? {'✓ YES' if facilities_code_unique == facilities_code_total else '✗ NO - duplicates found'}")

if facilities_code_unique < facilities_code_total:
    duplicate_codes = health_facility_type.groupby('Facilities_Code').filter(lambda x: len(x) > 1)
    print(f"\nDuplicate Facilities_Code analysis:")
    dup_summary = health_facility_type.groupby('Facilities_Code').size().sort_values(ascending=False)
    print(f"  Total duplicate codes: {len(dup_summary[dup_summary > 1])}")
    print(f"  Max duplicates per code: {dup_summary.max()}")
    
    print(f"\nSample duplicate codes (first 5):")
    for code, count in dup_summary[dup_summary > 1].head(5).items():
        group = health_facility_type[health_facility_type['Facilities_Code'] == code]
        print(f"  Code {code}: {count} records")
        print(f"    Facilities: {', '.join(group['Facility_Name'].head(3).tolist())}")
        if len(group) > 3:
            print(f"    ... and {len(group) - 3} more")

# Check if Facilities_Code follows a pattern
print(f"\nFacilities_Code Pattern Analysis:")
code_patterns = health_facility_type['Facilities_Code'].astype(str).str.len().value_counts().sort_index()
print(f"  Code length distribution:")
for length, count in code_patterns.items():
    print(f"    {length} digits: {count} records")

# Check if Facilities_Code relates to administrative codes
print(f"\nRelationship to Administrative Codes:")
sample_codes = health_facility_type[['Facilities_Code', 'State_Code', 'County_Code', 'Payam_Code ']].head(10)
print("Sample of Facilities_Code and administrative codes:")
for idx, row in sample_codes.iterrows():
    print(f"  {row['Facilities_Code']} → State: {row['State_Code']}, County: {row['County_Code']}, Payam: {row['Payam_Code ']}")

# Check if code prefix matches State_Code
health_facility_type['Code_prefix'] = health_facility_type['Facilities_Code'].astype(str).str[:4]
state_code_match = (health_facility_type['Code_prefix'] == health_facility_type['State_Code']).sum()
print(f"\nFacilities_Code prefix matches State_Code: {state_code_match:,} ({state_code_match/len(health_facility_type)*100:.1f}%)")

# Show mismatches
if state_code_match < len(health_facility_type):
    mismatches = health_facility_type[health_facility_type['Code_prefix'] != health_facility_type['State_Code']]
    print(f"Sample mismatches (first 5):")
    for idx, row in mismatches.head(5).iterrows():
        print(f"  {row['Facilities_Code']} → State_Code: {row['State_Code']} (expected prefix: {row['State_Code']})")

FACILITY IDENTITY INVESTIGATION: Facilities_Code

Dataset B: Facilities_Code Analysis
----------------------------------------
Total rows: 1,513
Unique codes: 1,511
Duplicate rate: 0.1%
Is Facilities_Code a true business key? ✗ NO - duplicates found

Duplicate Facilities_Code analysis:
  Total duplicate codes: 2
  Max duplicates per code: 2

Sample duplicate codes (first 5):
  Code 75020101: 2 records
    Facilities: Panyang PHCU, Abiemnom PHCC
  Code 92040402: 2 records
    Facilities: Tore PHCC, Kundru PHCU

Facilities_Code Pattern Analysis:
  Code length distribution:
    8 digits: 1513 records

Relationship to Administrative Codes:
Sample of Facilities_Code and administrative codes:
  71010101 → State: SS07, County: SS0711, Payam: SS071104
  71010201 → State: SS07, County: SS0711, Payam: SS071101
  71010202 → State: SS07, County: SS0711, Payam: SS071101
  71010203 → State: SS07, County: SS0711, Payam: SS071101
  71010204 → State: SS07, County: SS0711, Payam: SS071101
  71010301 → S

In [17]:
# CELL 10: Coordinate Comparison
import numpy as np 

print("="*80)
print("COORDINATE COMPARISON ANALYSIS")
print("="*80)

# 1. Coordinate coverage summary
print("\nCoordinate Coverage Summary:")
print("-" * 40)

coords_complete_a = (~health_facilities['missing_coordinates']).sum()
coords_missing_a = health_facilities['missing_coordinates'].sum()
print(f"Dataset A:")
print(f"  Facilities with coordinates: {coords_complete_a:,} ({coords_complete_a/len(health_facilities)*100:.1f}%)")
print(f"  Missing coordinates: {coords_missing_a:,} ({coords_missing_a/len(health_facilities)*100:.1f}%)")

coords_complete_b = (~health_facility_type['missing_coordinates']).sum()
coords_missing_b = health_facility_type['missing_coordinates'].sum()
print(f"\nDataset B:")
print(f"  Facilities with coordinates: {coords_complete_b:,} ({coords_complete_b/len(health_facility_type)*100:.1f}%)")
print(f"  Missing coordinates: {coords_missing_b:,} ({coords_missing_b/len(health_facility_type)*100:.1f}%)")

# 2. Compare coordinates for common facilities
print("\n" + "="*40)
print("Coordinate Comparison for Common Facilities")
print("-" * 40)

# Get common facilities with coordinates in both datasets
common_with_coords = []
for name in list(common_facilities)[:1000]:  # Sample for performance
    row_a = health_facilities[health_facilities['facility_name_clean'] == name]
    row_b = health_facility_type[health_facility_type['facility_name_clean'] == name]
    
    if len(row_a) > 0 and len(row_b) > 0:
        has_coords_a = not row_a.iloc[0]['missing_coordinates']
        has_coords_b = not row_b.iloc[0]['missing_coordinates']
        if has_coords_a and has_coords_b:
            common_with_coords.append({
                'name': name,
                'lat_a': row_a.iloc[0]['latitude'],
                'lon_a': row_a.iloc[0]['longitude'],
                'lat_b': row_b.iloc[0]['Latitude'],
                'lon_b': row_b.iloc[0]['Longitude']
            })

coords_df = pd.DataFrame(common_with_coords)
print(f"Common facilities with coordinates in both datasets: {len(coords_df):,}")

if len(coords_df) > 0:
    # Calculate coordinate differences
    coords_df['lat_diff'] = coords_df['lat_a'] - coords_df['lat_b']
    coords_df['lon_diff'] = coords_df['lon_a'] - coords_df['lon_b']
    
    # Calculate distance in km (approximate)
    coords_df['distance_km'] = np.sqrt(
        (coords_df['lat_diff'] * 111.32)**2 + 
        (coords_df['lon_diff'] * 111.32 * np.cos(np.radians(coords_df['lat_a'])))**2
    )
    
    print(f"\nCoordinate Differences Summary:")
    print(f"  Mean latitude difference: {coords_df['lat_diff'].mean():.4f}°")
    print(f"  Mean longitude difference: {coords_df['lon_diff'].mean():.4f}°")
    print(f"  Mean distance: {coords_df['distance_km'].mean():.2f} km")
    print(f"  Median distance: {coords_df['distance_km'].median():.2f} km")
    print(f"  Max distance: {coords_df['distance_km'].max():.2f} km")
    print(f"  Std deviation: {coords_df['distance_km'].std():.2f} km")
    
    # Facilities with exact match (within 0.001 km)
    exact_match = coords_df[coords_df['distance_km'] < 0.001]
    print(f"\n  Facilities with exact coordinate match: {len(exact_match)} ({len(exact_match)/len(coords_df)*100:.1f}%)")
    
    # Facilities with minor differences (< 1km)
    minor_diff = coords_df[(coords_df['distance_km'] >= 0.001) & (coords_df['distance_km'] < 1)]
    print(f"  Facilities with minor differences (<1km): {len(minor_diff)} ({len(minor_diff)/len(coords_df)*100:.1f}%)")
    
    # Facilities with major differences (>= 5km)
    major_diff = coords_df[coords_df['distance_km'] >= 5]
    print(f"  Facilities with major differences (>=5km): {len(major_diff)} ({len(major_diff)/len(coords_df)*100:.1f}%)")
    
    if len(major_diff) > 0:
        print(f"\nSample facilities with major coordinate disagreements (>5km):")
        for idx, row in major_diff.head(10).iterrows():
            print(f"  {row['name']}: {row['distance_km']:.1f}km difference")
            print(f"    Dataset A: ({row['lat_a']:.4f}, {row['lon_a']:.4f})")
            print(f"    Dataset B: ({row['lat_b']:.4f}, {row['lon_b']:.4f})")

# 3. Duplicate coordinate detection within datasets
print("\n" + "="*40)
print("Duplicate Coordinate Detection")
print("-" * 40)

def find_duplicate_coordinates(df, lat_col, lon_col, tolerance=0.0001):
    """Find facilities that share coordinates within a small tolerance."""
    # Round coordinates to reduce floating point issues
    df_rounded = df.copy()
    df_rounded['lat_rounded'] = df_rounded[lat_col].round(4)
    df_rounded['lon_rounded'] = df_rounded[lon_col].round(4)
    
    coords = df_rounded.groupby(['lat_rounded', 'lon_rounded']).size().reset_index(name='count')
    duplicates = coords[coords['count'] > 1]
    return duplicates

# Dataset A duplicates
df_a_with_coords = health_facilities[~health_facilities['missing_coordinates']].copy()
dups_a = find_duplicate_coordinates(df_a_with_coords, 'latitude', 'longitude')
print(f"Dataset A: {len(dups_a)} coordinate clusters with duplicate facilities")
if len(dups_a) > 0:
    print(f"  Max facilities per coordinate: {dups_a['count'].max()}")
    print("  Sample clusters:")
    for idx, row in dups_a.head(5).iterrows():
        print(f"    ({row['lat_rounded']:.4f}, {row['lon_rounded']:.4f}): {row['count']} facilities")

# Dataset B duplicates
df_b_with_coords = health_facility_type[~health_facility_type['missing_coordinates']].copy()
dups_b = find_duplicate_coordinates(df_b_with_coords, 'Latitude', 'Longitude')
print(f"\nDataset B: {len(dups_b)} coordinate clusters with duplicate facilities")
if len(dups_b) > 0:
    print(f"  Max facilities per coordinate: {dups_b['count'].max()}")
    print("  Sample clusters:")
    for idx, row in dups_b.head(5).iterrows():
        print(f"    ({row['lat_rounded']:.4f}, {row['lon_rounded']:.4f}): {row['count']} facilities")

COORDINATE COMPARISON ANALYSIS

Coordinate Coverage Summary:
----------------------------------------
Dataset A:
  Facilities with coordinates: 1,875 (94.3%)
  Missing coordinates: 113 (5.7%)

Dataset B:
  Facilities with coordinates: 1,288 (85.1%)
  Missing coordinates: 225 (14.9%)

Coordinate Comparison for Common Facilities
----------------------------------------
Common facilities with coordinates in both datasets: 911

Coordinate Differences Summary:
  Mean latitude difference: -0.0049°
  Mean longitude difference: 0.0063°
  Mean distance: 3.64 km
  Median distance: 0.00 km
  Max distance: 636.24 km
  Std deviation: 29.67 km

  Facilities with exact coordinate match: 645 (70.8%)
  Facilities with minor differences (<1km): 195 (21.4%)
  Facilities with major differences (>=5km): 49 (5.4%)

Sample facilities with major coordinate disagreements (>5km):
  GABAT PHCU: 47.7km difference
    Dataset A: (5.3506, 30.3254)
    Dataset B: (5.1506, 30.7056)
  PILIENY PHCU: 10.3km difference
 

In [18]:
# CELL 11: Text Consistency Investigation
print("="*80)
print("TEXT CONSISTENCY INVESTIGATION")
print("="*80)

# Analyze text quality in administrative columns
print("\nAdministrative Name Analysis:")
print("-" * 40)

# Check casing patterns
print("Casing Patterns:")

# Dataset A
print("\nDataset A:")
print(f"  State names - all lowercase: {health_facilities['state'].str.islower().all()}")
print(f"  County names - all lowercase: {health_facilities['county'].str.islower().all()}")
print(f"  Payam names - all lowercase: {health_facilities['payam'].str.islower().all()}")
print(f"  Facility names - mixed case: {not health_facilities['facility_name'].str.islower().all()}")

# Dataset B
print("\nDataset B:")
print(f"  State names - proper case: {not health_facility_type['State'].str.islower().all()}")
print(f"  County names - proper case: {not health_facility_type['County'].str.islower().all()}")
print(f"  Payam names - mixed case: {not health_facility_type['Payam'].str.islower().all()}")
print(f"  Facility names - mixed case: {not health_facility_type['Facility_Name'].str.islower().all()}")

# Check for whitespace issues
print("\n" + "="*40)
print("Whitespace Issues:")
print("-" * 40)

def check_whitespace(df, df_name, columns):
    print(f"\n{df_name}:")
    for col in columns:
        if col not in df.columns:
            continue
        leading = df[col].astype(str).str.startswith(' ').sum()
        trailing = df[col].astype(str).str.endswith(' ').sum()
        multiple = df[col].astype(str).str.contains('  ').sum()
        total = leading + trailing + multiple
        if total > 0:
            print(f"  {col}: {total} issues (leading: {leading}, trailing: {trailing}, multiple: {multiple})")

# Check administrative columns
check_whitespace(health_facilities, 'Dataset A', ['state', 'county', 'payam', 'facility_name'])
check_whitespace(health_facility_type, 'Dataset B', ['State', 'County', 'Payam', 'Facility_Name', 'Facility_type'])

# Check for column name issues
print("\n" + "="*40)
print("Column Name Issues:")
print("-" * 40)
print(f"Dataset B has column: 'Payam_Code ' (trailing space) → This should be renamed to 'Payam_Code'")

# Analyze facility type abbreviations (Dataset B)
print("\n" + "="*40)
print("Facility Type Analysis (Dataset B):")
print("-" * 40)

facility_types = health_facility_type['Facility_type'].value_counts()
print(f"Facility type distribution:")
for ftype, count in facility_types.items():
    print(f"  {ftype}: {count} ({count/len(health_facility_type)*100:.1f}%)")

# Look for PHC patterns
print(f"\nPHC-related facility types:")
phc_types = health_facility_type[health_facility_type['Facility_type'].str.contains('PHC', na=False)]
print(f"  Total PHC-related: {len(phc_types)} ({len(phc_types)/len(health_facility_type)*100:.1f}%)")
phc_patterns = phc_types['Facility_type'].value_counts()
for pattern, count in phc_patterns.items():
    print(f"    {pattern}: {count}")

# Check for common abbreviations and variants
print(f"\nCommon naming patterns in facilities:")
# Check if facility names follow consistent pattern (e.g., "NAME TYPE")
facility_names = health_facility_type['Facility_Name'].astype(str)
has_type_in_name = facility_names.str.contains('PHCU|PHCC|Hospital|Clinic|Health', case=False, na=False)
print(f"  Facilities with type indicator in name: {has_type_in_name.sum()} ({has_type_in_name.sum()/len(health_facility_type)*100:.1f}%)")

# Sample facilities without type indicator
no_type = health_facility_type[~has_type_in_name]
if len(no_type) > 0:
    print(f"\n  Sample facilities without type indicator:")
    for name in no_type['Facility_Name'].head(5).tolist():
        print(f"    {name}")

# Check Dataset A for similar patterns
facility_names_a = health_facilities['facility_name'].astype(str)
has_type_in_name_a = facility_names_a.str.contains('PHCU|PHCC|Hospital|Clinic|Health', case=False, na=False)
print(f"\nDataset A - Facilities with type indicator in name: {has_type_in_name_a.sum()} ({has_type_in_name_a.sum()/len(health_facilities)*100:.1f}%)")

TEXT CONSISTENCY INVESTIGATION

Administrative Name Analysis:
----------------------------------------
Casing Patterns:

Dataset A:
  State names - all lowercase: True
  County names - all lowercase: True
  Payam names - all lowercase: True
  Facility names - mixed case: False

Dataset B:
  State names - proper case: True
  County names - proper case: True
  Payam names - mixed case: True
  Facility names - mixed case: True

Whitespace Issues:
----------------------------------------

Dataset A:

Dataset B:
  County: 20 issues (leading: 0, trailing: 20, multiple: 0)
  Facility_Name: 12 issues (leading: 0, trailing: 8, multiple: 4)

Column Name Issues:
----------------------------------------
Dataset B has column: 'Payam_Code ' (trailing space) → This should be renamed to 'Payam_Code'

Facility Type Analysis (Dataset B):
----------------------------------------
Facility type distribution:
  PHCU: 1065 (70.4%)
  PHCC: 373 (24.7%)
  Hospital: 75 (5.0%)

PHC-related facility types:
  Total

In [20]:
# CELL 13: Merge Feasibility Analysis
print("="*80)
print("MERGE FEASIBILITY ANALYSIS")
print("="*80)

# 1. Test different join strategies
print("\nTesting Join Strategies:")
print("-" * 40)

# Strategy 1: Facility Name only
print("\nStrategy 1: Facility Name (case-insensitive)")
join_name = pd.merge(
    health_facilities,
    health_facility_type,
    left_on='facility_name_clean',
    right_on='facility_name_clean',
    how='inner',
    suffixes=('_A', '_B')
)
print(f"  Matched records: {len(join_name):,}")
print(f"  Match rate (of A): {len(join_name)/len(health_facilities)*100:.1f}%")
print(f"  Match rate (of B): {len(join_name)/len(health_facility_type)*100:.1f}%")

# Strategy 2: Facility Name + State
print("\nStrategy 2: Facility Name + State (case-insensitive)")
# Create composite keys with standardized state
health_facilities['state_key'] = health_facilities['state'].str.upper()
health_facility_type['state_key'] = health_facility_type['State'].str.upper()

join_name_state = pd.merge(
    health_facilities,
    health_facility_type,
    left_on=['facility_name_clean', 'state_key'],
    right_on=['facility_name_clean', 'state_key'],
    how='inner',
    suffixes=('_A', '_B')
)
print(f"  Matched records: {len(join_name_state):,}")
print(f"  Match rate (of A): {len(join_name_state)/len(health_facilities)*100:.1f}%")
print(f"  Match rate (of B): {len(join_name_state)/len(health_facility_type)*100:.1f}%")

# Strategy 3: Facility Name + State + County
print("\nStrategy 3: Facility Name + State + County")
health_facilities['county_key'] = health_facilities['county'].str.upper()
health_facility_type['county_key'] = health_facility_type['County'].str.upper()

join_name_state_county = pd.merge(
    health_facilities,
    health_facility_type,
    left_on=['facility_name_clean', 'state_key', 'county_key'],
    right_on=['facility_name_clean', 'state_key', 'county_key'],
    how='inner',
    suffixes=('_A', '_B')
)
print(f"  Matched records: {len(join_name_state_county):,}")
print(f"  Match rate (of A): {len(join_name_state_county)/len(health_facilities)*100:.1f}%")
print(f"  Match rate (of B): {len(join_name_state_county)/len(health_facility_type)*100:.1f}%")

# 2. Analyze unmatched records
print("\n" + "="*40)
print("Unmatched Records Analysis:")
print("-" * 40)

# Find records that didn't match using Strategy 2 (Name + State)
unmatched_a = health_facilities[
    ~health_facilities.set_index(['facility_name_clean', 'state_key']).index.isin(
        health_facility_type.set_index(['facility_name_clean', 'state_key']).index
    )
]
unmatched_b = health_facility_type[
    ~health_facility_type.set_index(['facility_name_clean', 'state_key']).index.isin(
        health_facilities.set_index(['facility_name_clean', 'state_key']).index
    )
]

print(f"Unmatched in Dataset A: {len(unmatched_a):,}")
print(f"Unmatched in Dataset B: {len(unmatched_b):,}")

if len(unmatched_a) > 0:
    print(f"\nSample unmatched records from Dataset A (first 5):")
    for idx, row in unmatched_a.head(5).iterrows():
        print(f"  {row['facility_name_clean']} - {row['state']}")

if len(unmatched_b) > 0:
    print(f"\nSample unmatched records from Dataset B (first 5):")
    for idx, row in unmatched_b.head(5).iterrows():
        print(f"  {row['facility_name_clean']} - {row['State']}")

# 3. Duplicate risk assessment
print("\n" + "="*40)
print("Duplicate Risk Assessment:")
print("-" * 40)

# Check for multiple matches
duplicate_matches = join_name_state.groupby('facility_name_clean').size()
multiple_matches = duplicate_matches[duplicate_matches > 1]
print(f"Facilities with multiple potential matches: {len(multiple_matches)}")
if len(multiple_matches) > 0:
    print(f"  Max matches per facility: {multiple_matches.max()}")
    print(f"  Sample facilities with multiple matches:")
    for name, count in multiple_matches.head(5).items():
        print(f"    {name}: {count} matches")

# 4. Merge recommendation
print("\n" + "="*40)
print("Merge Strategy Recommendation:")
print("-" * 40)

print("Recommended Join Key: Facility Name + State (case-insensitive)")
print(f"  Expected match rate: {len(join_name_state)/len(health_facilities)*100:.1f}% of Dataset A")
print(f"  Match quality: {len(join_name_state[join_name_state['state_key'] == join_name_state['state_key']]):,} records")
print(f"  Duplicate risk: {'LOW' if len(multiple_matches) == 0 else 'MODERATE'}")

print("\nMerge Confidence Levels:")
print("  HIGH confidence: Facility Name + State + County match (strict)")
print("  MEDIUM confidence: Facility Name + State match (recommended)")
print("  LOW confidence: Facility Name only match (potential false positives)")

print("\nMerge Risks:")
print("  1. Abyei Administrative Area has inconsistent naming between datasets")
print("  2. State abbreviations in Dataset A vs full names in Dataset B")
print("  3. 49 facilities have coordinate disagreements >5km")
print("  4. Some duplicate facilities share the same code")

MERGE FEASIBILITY ANALYSIS

Testing Join Strategies:
----------------------------------------

Strategy 1: Facility Name (case-insensitive)
  Matched records: 1,344
  Match rate (of A): 67.6%
  Match rate (of B): 88.8%

Strategy 2: Facility Name + State (case-insensitive)
  Matched records: 533
  Match rate (of A): 26.8%
  Match rate (of B): 35.2%

Strategy 3: Facility Name + State + County
  Matched records: 487
  Match rate (of A): 24.5%
  Match rate (of B): 32.2%

Unmatched Records Analysis:
----------------------------------------
Unmatched in Dataset A: 1,464
Unmatched in Dataset B: 987

Sample unmatched records from Dataset A (first 5):
  ABYEI CIVIL HOSPITAL - abyei region
  ABYEI PHCC - abyei region
  AGANY TOK PHCU - abyei region
  AGOK PHCC - abyei region
  AMENTH BEK HOSPITAL - abyei region

Sample unmatched records from Dataset B (first 5):
  MORRO PHCU - Upper Nile
  NYANWAR PHCU - Upper Nile
  AKOK PHCU - Upper Nile
  DETHWOK PHCU - Upper Nile
  ABUROC 1 PHCC - Upper Nile

In [21]:
# CELL 14: Business Rule Validation
print("="*80)
print("BUSINESS RULE VALIDATION")
print("="*80)

# Rule 1: Every facility should have coordinates
print("\nRule 1: Every facility should have coordinates")
print("-" * 40)
missing_coords_a = health_facilities['missing_coordinates'].sum()
missing_coords_b = health_facility_type['missing_coordinates'].sum()
print(f"Dataset A: {missing_coords_a:,} facilities missing coordinates ({missing_coords_a/len(health_facilities)*100:.1f}%) - VIOLATION")
print(f"Dataset B: {missing_coords_b:,} facilities missing coordinates ({missing_coords_b/len(health_facility_type)*100:.1f}%) - VIOLATION")

# Rule 2: Every facility should belong to exactly one Payam
print("\nRule 2: Every facility should belong to exactly one Payam")
print("-" * 40)
payam_null_a = health_facilities['payam'].isna().sum()
payam_null_b = health_facility_type['Payam'].isna().sum()
print(f"Dataset A: {payam_null_a} facilities missing Payam ({payam_null_a/len(health_facilities)*100:.1f}%) - VIOLATION")
print(f"Dataset B: {payam_null_b} facilities missing Payam - ✓ PASS")

# Rule 3: Every State should have exactly one State_Code
print("\nRule 3: Every State should have exactly one State_Code")
print("-" * 40)
state_code_dups_a = health_facilities.groupby('state')['state_code'].nunique()
state_code_dups_b = health_facility_type.groupby('State')['State_Code'].nunique()
violations_a = (state_code_dups_a > 1).sum()
violations_b = (state_code_dups_b > 1).sum()
print(f"Dataset A: {violations_a} states with multiple codes - ✓ PASS")
print(f"Dataset B: {violations_b} states with multiple codes - VIOLATION")
if violations_b > 0:
    print("  Violating states:")
    for state, count in state_code_dups_b[state_code_dups_b > 1].items():
        codes = health_facility_type[health_facility_type['State'] == state]['State_Code'].unique()
        print(f"    {state}: {codes}")

# Rule 4: Every County should belong to exactly one State
print("\nRule 4: Every County should belong to exactly one State")
print("-" * 40)
county_state_a = health_facilities.groupby('county')['state'].nunique()
county_state_b = health_facility_type.groupby('County')['State'].nunique()
violations_a = (county_state_a > 1).sum()
violations_b = (county_state_b > 1).sum()
print(f"Dataset A: {violations_a} counties spanning multiple states - ✓ PASS")
print(f"Dataset B: {violations_b} counties spanning multiple states - ✓ PASS")

# Rule 5: Every Payam should belong to exactly one County
print("\nRule 5: Every Payam should belong to exactly one County")
print("-" * 40)
payam_county_a = health_facilities.groupby('payam')['county'].nunique()
payam_county_b = health_facility_type.groupby('Payam')['County'].nunique()
violations_a = (payam_county_a > 1).sum()
violations_b = (payam_county_b > 1).sum()
print(f"Dataset A: {violations_a} payams spanning multiple counties - VIOLATION")
print(f"Dataset B: {violations_b} payams spanning multiple counties - VIOLATION")

# Rule 6: Facility names should be unique within a Payam
print("\nRule 6: Facility names should be unique within a Payam")
print("-" * 40)
# Dataset A
dup_names_a = health_facilities.groupby(['payam', 'facility_name']).size()
dup_names_a = dup_names_a[dup_names_a > 1]
print(f"Dataset A: {len(dup_names_a)} duplicate facility names within same payam")

# Dataset B
dup_names_b = health_facility_type.groupby(['Payam', 'Facility_Name']).size()
dup_names_b = dup_names_b[dup_names_b > 1]
print(f"Dataset B: {len(dup_names_b)} duplicate facility names within same payam")

# Rule 7: Coordinates should be within South Sudan bounds
print("\nRule 7: Coordinates should be within South Sudan bounds")
print("-" * 40)
# South Sudan approximate bounds: lat 3.5-12.2, lon 24-36
valid_coords_a = health_facilities[
    ~health_facilities['missing_coordinates'] &
    (health_facilities['latitude'].between(3.5, 12.2)) &
    (health_facilities['longitude'].between(24, 36))
]
valid_coords_b = health_facility_type[
    ~health_facility_type['missing_coordinates'] &
    (health_facility_type['Latitude'].between(3.5, 12.2)) &
    (health_facility_type['Longitude'].between(24, 36))
]
print(f"Dataset A: {len(valid_coords_a)} of {coords_complete_a} coordinates valid ({len(valid_coords_a)/coords_complete_a*100:.1f}%)")
print(f"Dataset B: {len(valid_coords_b)} of {coords_complete_b} coordinates valid ({len(valid_coords_b)/coords_complete_b*100:.1f}%)")

# Rule 8: Facility type should be one of valid types (Dataset B only)
print("\nRule 8: Facility type should be one of valid types")
print("-" * 40)
valid_types = ['PHCU', 'PHCC', 'Hospital']
invalid_types = health_facility_type[~health_facility_type['Facility_type'].isin(valid_types)]
print(f"Dataset B: {len(invalid_types)} facilities with invalid types ({len(invalid_types)/len(health_facility_type)*100:.1f}%) - VIOLATION")

# Additional business rule: Facilities_Code should be unique
print("\nRule 9: Facilities_Code should be unique")
print("-" * 40)
code_dups = health_facility_type['Facilities_Code'].duplicated().sum()
print(f"Dataset B: {code_dups} duplicate Facilities_Code ({code_dups/len(health_facility_type)*100:.1f}%) - VIOLATION")

# Summary of violations
print("\n" + "="*40)
print("BUSINESS RULE VIOLATIONS SUMMARY:")
print("-" * 40)
violations = [
    ("Rule 1: All facilities have coordinates", "Dataset A: VIOLATION, Dataset B: VIOLATION"),
    ("Rule 2: All facilities have Payam", "Dataset A: VIOLATION, Dataset B: PASS"),
    ("Rule 3: One State_Code per State", "Dataset A: PASS, Dataset B: VIOLATION"),
    ("Rule 4: One State per County", "Dataset A: PASS, Dataset B: PASS"),
    ("Rule 5: One County per Payam", "Dataset A: VIOLATION, Dataset B: VIOLATION"),
    ("Rule 6: Unique facility names within Payam", f"Dataset A: {len(dup_names_a)} violations, Dataset B: {len(dup_names_b)} violations"),
    ("Rule 7: Valid coordinates", f"Dataset A: {len(valid_coords_a)}/{coords_complete_a}, Dataset B: {len(valid_coords_b)}/{coords_complete_b}"),
    ("Rule 8: Valid facility types", f"Dataset B: {len(invalid_types)} violations"),
    ("Rule 9: Unique Facilities_Code", f"Dataset B: {code_dups} violations")
]

for rule, status in violations:
    print(f"  {rule}: {status}")

BUSINESS RULE VALIDATION

Rule 1: Every facility should have coordinates
----------------------------------------
Dataset A: 113 facilities missing coordinates (5.7%) - VIOLATION
Dataset B: 225 facilities missing coordinates (14.9%) - VIOLATION

Rule 2: Every facility should belong to exactly one Payam
----------------------------------------
Dataset A: 23 facilities missing Payam (1.2%) - VIOLATION
Dataset B: 0 facilities missing Payam - ✓ PASS

Rule 3: Every State should have exactly one State_Code
----------------------------------------
Dataset A: 0 states with multiple codes - ✓ PASS
Dataset B: 1 states with multiple codes - VIOLATION
  Violating states:
    Abyei Adminstrative Area: ['SS00' 'SS01' 'SS02' 'SS03' 'SS04' 'SS05' 'SS06' 'SS07' 'SS08' 'SS09'
 'SS10' 'SS11' 'SS12' 'SS13' 'SS14' 'SS15' 'SS16' 'SS17' 'SS18']

Rule 4: Every County should belong to exactly one State
----------------------------------------
Dataset A: 0 counties spanning multiple states - ✓ PASS
Dataset B: 0

In [22]:
# CELL 15: Final Assessment Summary
print("="*80)
print("FINAL ASSESSMENT: DATA QUALITY INVESTIGATION SUMMARY")
print("="*80)

# Create summary table
summary_data = [
    {
        "Investigation": "Administrative Hierarchy",
        "Dataset A Result": "Good - All codes map 1:1, but all lowercase",
        "Dataset B Result": "CRITICAL - Abyei has 19 State_Code variants, 1 Payam code maps to 2 payams",
        "Severity": "HIGH",
        "Recommended Action": "Standardize state names and fix Abyei State_Code mapping"
    },
    {
        "Investigation": "Dataset Overlap",
        "Dataset A Result": f"63.5% of A in B ({len(only_in_a):,} unique)",
        "Dataset B Result": f"83.5% of B in A ({len(only_in_b):,} unique)",
        "Severity": "MEDIUM",
        "Recommended Action": "Investigate 711 facilities unique to A and 244 unique to B"
    },
    {
        "Investigation": "State Name Consistency",
        "Dataset A Result": "Uses abbreviations (cent eq, east eq, etc.)",
        "Dataset B Result": "Uses full names, but has 'UNITY' and 'Unity' variants",
        "Severity": "HIGH",
        "Recommended Action": "Create state name mapping dictionary for standardization"
    },
    {
        "Investigation": "Coordinate Coverage",
        "Dataset A Result": f"94.3% complete ({coords_complete_a:,} facilities)",
        "Dataset B Result": f"85.1% complete ({coords_complete_b:,} facilities)",
        "Severity": "MEDIUM",
        "Recommended Action": "Use Dataset A as primary coordinate source; investigate missing coords"
    },
    {
        "Investigation": "Coordinate Agreement",
        "Dataset A Result": "70.8% exact match with B",
        "Dataset B Result": "5.4% have >5km disagreement (49 facilities)",
        "Severity": "HIGH",
        "Recommended Action": "Investigate 49 facilities with major coordinate discrepancies"
    },
    {
        "Investigation": "Text Quality",
        "Dataset A Result": "All lowercase, good consistency",
        "Dataset B Result": "Mixed case, trailing spaces, column name issues",
        "Severity": "LOW",
        "Recommended Action": "Standardize casing and trim whitespace"
    },
    {
        "Investigation": "Facility Identity",
        "Dataset A Result": "Facility names mostly unique (75 dupes)",
        "Dataset B Result": "Facilities_Code 99.9% unique, but duplicates exist",
        "Severity": "MEDIUM",
        "Recommended Action": "Use Facility_Name + State as composite key"
    },
    {
        "Investigation": "Merge Feasibility",
        "Dataset A Result": f"26.8% match rate with B using Name+State",
        "Dataset B Result": f"35.2% match rate with A using Name+State",
        "Severity": "CRITICAL",
        "Recommended Action": "Low merge confidence - consider if datasets should be merged"
    },
    {
        "Investigation": "Business Rules",
        "Dataset A Result": "4 violations (coords, payam, county-payam)",
        "Dataset B Result": "6 violations (state code, county-payam, duplicate code)",
        "Severity": "HIGH",
        "Recommended Action": "Fix Abyei State_Code issue as top priority"
    }
]

summary_df = pd.DataFrame(summary_data)

# Display summary table
print("\nDATASET COMPARISON SUMMARY TABLE:")
print("-" * 80)
for idx, row in summary_df.iterrows():
    print(f"\n{row['Investigation']}:")
    print(f"  Dataset A: {row['Dataset A Result']}")
    print(f"  Dataset B: {row['Dataset B Result']}")
    print(f"  Severity: {row['Severity']}")
    print(f"  Action: {row['Recommended Action']}")

# Final verdict
print("\n" + "="*80)
print("FINAL VERDICT")
print("="*80)

print("\nWhich dataset should be the authoritative source?")
print("-" * 40)
print("RECOMMENDATION: Dataset A should be the primary authoritative source")
print("\nReasons:")
print("  1. More complete record count (1,988 vs 1,513)")
print("  2. Better coordinate coverage (94.3% vs 85.1%)")
print("  3. More consistent administrative coding (no multiple State_Code issues)")
print("  4. Cleaner text consistency (all lowercase)")
print("  5. Dataset B has critical State_Code mapping issues (Abyei problem)")

print("\nShould these datasets be merged?")
print("-" * 40)
print("RECOMMENDATION: Proceed with CAUTION or avoid merging entirely")
print("\nReasons:")
print("  1. Low match rate (only 26.8% using best join strategy)")
print("  2. Significant administrative differences (abbreviations vs full names)")
print("  3. Coordinate discrepancies for 49 facilities")
print("  4. Dataset B's Abyei State_Code issue makes matching unreliable")

print("\nAlternative Approach:")
print("  1. Keep datasets separate but cross-reference")
print("  2. Use Dataset A as the master facility list")
print("  3. Use Dataset B as supplementary source for Facility_Type only (75 facilities have this)")
print("  4. Investigate 711 facilities unique to A and 244 unique to B before any merge")

print("\n" + "="*40)
print("PRIORITY CLEANING ACTIONS (in order):")
print("-" * 40)
print("1. CRITICAL: Fix Abyei Administrative Area State_Code mapping in Dataset B")
print("2. HIGH: Standardize state names (Dataset A abbreviations → full names)")
print("3. HIGH: Investigate 49 facilities with coordinate disagreements >5km")
print("4. HIGH: Investigate why 711 facilities in Dataset A are not in Dataset B")
print("5. MEDIUM: Fix column name issue ('Payam_Code ' trailing space)")
print("6. MEDIUM: Standardize casing across both datasets")
print("7. MEDIUM: Resolve duplicate Facilities_Code issues")
print("8. LOW: Trim whitespace and clean text")

FINAL ASSESSMENT: DATA QUALITY INVESTIGATION SUMMARY

DATASET COMPARISON SUMMARY TABLE:
--------------------------------------------------------------------------------

Administrative Hierarchy:
  Dataset A: Good - All codes map 1:1, but all lowercase
  Dataset B: CRITICAL - Abyei has 19 State_Code variants, 1 Payam code maps to 2 payams
  Severity: HIGH
  Action: Standardize state names and fix Abyei State_Code mapping

Dataset Overlap:
  Dataset A: 63.5% of A in B (711 unique)
  Dataset B: 83.5% of B in A (244 unique)
  Severity: MEDIUM
  Action: Investigate 711 facilities unique to A and 244 unique to B

State Name Consistency:
  Dataset A: Uses abbreviations (cent eq, east eq, etc.)
  Dataset B: Uses full names, but has 'UNITY' and 'Unity' variants
  Severity: HIGH
  Action: Create state name mapping dictionary for standardization

Coordinate Coverage:
  Dataset A: 94.3% complete (1,875 facilities)
  Dataset B: 85.1% complete (1,288 facilities)
  Severity: MEDIUM
  Action: Use Dat

In [23]:
# CELL 16: Investigation Conclusion & Next Steps
print("="*80)
print("INVESTIGATION CONCLUSION")
print("="*80)

print("\nInvestigation Complete: 16 Cells Executed")
print("-" * 40)
print("Total Investigations Performed:")
print("  1. Initial Data Overview")
print("  2. Administrative Hierarchy - State Level")
print("  3. Administrative Hierarchy - County Level")
print("  4. Administrative Hierarchy - Payam Level")
print("  5. Cross-Table Comparison - Facility Overlap")
print("  6. Cross-Table Comparison - Administrative Disagreements")
print("  7. Case-Insensitive Administrative Comparison")
print("  8. State Name Standardization Analysis")
print("  9. Facility Identity - Facilities_Code Analysis")
print(" 10. Coordinate Comparison")
print(" 11. Text Consistency Investigation")
print(" 12. Coverage Analysis")
print(" 13. Merge Feasibility Analysis")
print(" 14. Business Rule Validation")
print(" 15. Final Assessment Summary")
print(" 16. Investigation Conclusion")

print("\n" + "="*80)
print("KEY FINDINGS SUMMARY")
print("="*80)

findings = [
    ("Dataset A is the better authoritative source", "More records, better coverage, cleaner data"),
    ("Dataset B has critical State_Code issues", "Abyei Administrative Area has 19 different codes"),
    ("Low merge confidence", "Only 26.8% match rate using best join strategy"),
    ("Major state name differences", "Abbreviations in A vs full names in B"),
    ("49 facilities have coordinate disagreements >5km", "Requires investigation before any spatial analysis"),
    ("711 facilities unique to Dataset A", "Need to verify if these are real facilities or data entry errors"),
    ("244 facilities unique to Dataset B", "May be newer facilities not yet in Dataset A")
]

print("\nTop 7 Critical Findings:")
for i, (finding, detail) in enumerate(findings, 1):
    print(f"\n{i}. {finding}")
    print(f"   → {detail}")

print("\n" + "="*80)
print("RECOMMENDED NEXT STEPS")
print("="*80)

steps = [
    ("Immediate", "Fix Abyei State_Code issue in Dataset B (critical)"),
    ("Immediate", "Create state name mapping dictionary for standardization"),
    ("Immediate", "Investigate 49 facilities with coordinate discrepancies >5km"),
    ("Short-term", "Verify 711 facilities unique to Dataset A"),
    ("Short-term", "Investigate 244 facilities unique to Dataset B"),
    ("Short-term", "Clean text quality issues (whitespace, casing)"),
    ("Medium-term", "Decide on merge strategy vs. keeping datasets separate"),
    ("Medium-term", "Create unified facility registry with authoritative source tags"),
    ("Long-term", "Establish data governance for future facility data collection")
]

print("\nPrioritized Action Plan:")
for priority, action in steps:
    print(f"  [{priority}] {action}")

print("\n" + "="*80)
print("INVESTIGATION NOTEBOOK COMPLETE")
print("="*80)
print("\nThis notebook has identified data quality issues and provided recommendations.")
print("No cleaning has been performed - all investigations were for discovery only.")
print("\nProceed to cleaning phase with these findings as your guide.")

INVESTIGATION CONCLUSION

Investigation Complete: 16 Cells Executed
----------------------------------------
Total Investigations Performed:
  1. Initial Data Overview
  2. Administrative Hierarchy - State Level
  3. Administrative Hierarchy - County Level
  4. Administrative Hierarchy - Payam Level
  5. Cross-Table Comparison - Facility Overlap
  6. Cross-Table Comparison - Administrative Disagreements
  7. Case-Insensitive Administrative Comparison
  8. State Name Standardization Analysis
  9. Facility Identity - Facilities_Code Analysis
 10. Coordinate Comparison
 11. Text Consistency Investigation
 12. Coverage Analysis
 13. Merge Feasibility Analysis
 14. Business Rule Validation
 15. Final Assessment Summary
 16. Investigation Conclusion

KEY FINDINGS SUMMARY

Top 7 Critical Findings:

1. Dataset A is the better authoritative source
   → More records, better coverage, cleaner data

2. Dataset B has critical State_Code issues
   → Abyei Administrative Area has 19 different codes

